In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

model = mlflow.sklearn.load_model(
    "models:/fraud_demo.transactions.fraud_classifier@Champion"
)

In [0]:
raw_df = spark.table("fraud_demo.transactions.raw").toPandas()
features_df = (spark.table("fraud_demo.transactions.user_features")
               .toPandas())

training_df = raw_df.merge(features_df, on="user_id", how="left")

feature_cols = [
    "amount", "hour",
    "txn_count_7d", "avg_amount_7d",
    "max_amount_7d", "std_amount_7d",
    "unknown_merchant_count"
]

X = training_df[feature_cols]
y = training_df["is_fraud"]


In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [0]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay


In [0]:
client = mlflow.tracking.MlflowClient()

latest = client.search_model_versions(
    filter_string="name='fraud_demo.transactions.fraud_classifier'"
)
run_id = latest[0].run_id

In [0]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

RocCurveDisplay.from_estimator(
    model, X_test, y_test, ax=axes[0], name="GBT fraud model")
axes[0].set_title("ROC curve")

ConfusionMatrixDisplay.from_estimator(
    model, X_test, y_test, ax=axes[1],
    display_labels=["legit","fraud"])
axes[1].set_title("Confusion matrix")

plt.tight_layout()

with mlflow.start_run(run_id=run_id):
    mlflow.log_figure(fig, "evaluation_plots.png")

display(fig)